# OBE dataset generation

This notebook is a Colab-friendly front end to `scripts/generate_dataset.py`. The script renders each answer PDF once, expands each answer into criterion-level records, checks for cross-split leakage, and either uploads a private Hugging Face dataset or saves it locally.

Expected input layout:

```text
Processed_Dataset/
├── train.json
├── val.json
├── test.json
└── files/
    └── *.pdf
```


In [ ]:
%pip install -q "PyMuPDF>=1.24,<2" "Pillow>=10,<13" "datasets>=2.19,<5" "huggingface_hub>=0.23,<1" "tqdm>=4.66,<5"


## Configure the run

Update the paths and repository ID. Leave `UPLOAD_TO_HUB` as `False` to build a local DatasetDict instead.


In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Processed_Dataset")
HF_REPO = "your-account/obe-exam-grading"
LOCAL_OUTPUT = Path("/content/obe-exam-grading")
UPLOAD_TO_HUB = False

DPI = 200
JPEG_QUALITY = 85
MAX_LONG_SIDE = 1280
MAX_HEIGHT = 3072
FALLBACK_CRITERION_MAX = 4.0


## Authenticate only when uploading

The login widget stores the credential in the notebook session. No token is embedded in this notebook or passed on a command line.


In [ ]:
if UPLOAD_TO_HUB:
    from huggingface_hub import notebook_login
    notebook_login()


## Build the dataset

Run this cell from the repository checkout. In Colab, clone or upload the repository first and change `REPO_ROOT` if needed.


In [ ]:
import subprocess
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
script = REPO_ROOT / "scripts" / "generate_dataset.py"
command = [
    sys.executable, str(script),
    "--data-dir", str(DATA_DIR),
    "--dpi", str(DPI),
    "--jpeg-quality", str(JPEG_QUALITY),
    "--max-long-side", str(MAX_LONG_SIDE),
    "--max-height", str(MAX_HEIGHT),
    "--fallback-criterion-max", str(FALLBACK_CRITERION_MAX),
]
command += (["--repo-id", HF_REPO] if UPLOAD_TO_HUB
            else ["--output-dir", str(LOCAL_OUTPUT)])
subprocess.run(command, cwd=REPO_ROOT, check=True)


## Verify a local build

This optional cell loads the saved DatasetDict and checks that the first image decodes correctly.


In [ ]:
if not UPLOAD_TO_HUB:
    from datasets import load_from_disk
    generated = load_from_disk(str(LOCAL_OUTPUT))
    print(generated)
    sample = generated["train"][0]
    print(sample["example_id"], sample["image"].size, sample["criterion_name"])
